In [ ]:
import pandas as pd

# Load the files with proper header handling
df1 = pd.read_excel('data/data.xlsx', sheet_name='Sheet1', header=2)  # Skip first 2 rows
df2 = pd.read_csv('data/df9dc704-0808-4e27-81db-ee0c32860c9b.csv')
df3 = pd.read_csv('data/268a42f0-3ac9-46de-af9d-71fdbcc69aeb.csv')

# Standardize column names across all datasets
column_mapping = {
    'SpatialDimValueCode': 'LocationCode',
    'Location': 'Country',
    'Period type': 'PeriodType',
    'Location type': 'LocationType',
    'Dim1 type': 'Dim1Type',
    'Dim2 type': 'Dim2Type',
    'Dim3 type': 'Dim3Type'
}

# Apply renaming to all DataFrames
for df in [df1, df2, df3]:
    df.rename(columns={k:v for k,v in column_mapping.items() if k in df.columns}, inplace=True)

# Verify we have consistent columns before concatenation
print("DF1 columns:", df1.columns.tolist())
print("DF2 columns:", df2.columns.tolist())
print("DF3 columns:", df3.columns.tolist())

# Combine all datasets
combined_df = pd.concat([df1, df2, df3], ignore_index=True)

# --- Data Cleaning Steps ---

# 1. Remove exact duplicates
combined_df = combined_df.drop_duplicates()

# 2. Remove near-duplicates (keeping first occurrence)
combined_df = combined_df.drop_duplicates(
    subset=['Country', 'IndicatorCode', 'Period'],
    keep='first'
)

# 3. Handle missing values
# Drop rows where critical columns are missing
combined_df = combined_df.dropna(subset=['Country', 'IndicatorCode'])

# Fill numeric missing values with 0
numeric_cols = combined_df.select_dtypes(include=['float64', 'int64']).columns
combined_df[numeric_cols] = combined_df[numeric_cols].fillna(0)

# 4. Standardize text data
text_cols = ['Country', 'Indicator', 'Value', 'Language']
for col in text_cols:
    if col in combined_df:
        combined_df[col] = combined_df[col].astype(str).str.strip().str.title()

# 5. Fix inconsistent values
combined_df.replace(['Not available', 'N/A', 'NA', 'nan', 'None'], pd.NA, inplace=True)

# 6. Handle date columns carefully to avoid timezone issues
if 'DateModified' in combined_df:
    # First convert all to strings to avoid mixed timezone issues
    combined_df['DateModified'] = combined_df['DateModified'].astype(str)
    # Then convert to datetime, coercing errors
    combined_df['DateModified'] = pd.to_datetime(combined_df['DateModified'], errors='coerce')

# 7. Drop unnecessary columns
columns_to_drop = ['FactComments', 'Dim1', 'Dim2', 'Dim3', 'Dim1ValueCode',
                  'Dim2ValueCode', 'Dim3ValueCode', 'DataSourceDimValueCode']
combined_df = combined_df.drop(columns=[col for col in columns_to_drop if col in combined_df])

# 8. Final cleanup - drop any remaining all-NA columns
combined_df = combined_df.dropna(axis=1, how='all')

# Save the cleaned data
combined_df.to_excel('data/cleaned_combined_data.xlsx', index=False)
print(f"✅ Data successfully cleaned and saved. Final shape: {combined_df.shape}")
print("Columns in final dataset:", combined_df.columns.tolist())

In [ ]:
# Calculating total number of columns:
print("Total columns in cleaned data:", len(combined_df.columns))
print("\nColumn names:")
for i, col in enumerate(combined_df.columns, 1):
    print(f"{i}. {col}")

In [ ]:
# Calculating total number of rows:
print("Total rows in cleaned data:", combined_df.shape[0])

In [ ]:
# Handle missing values
# Fill numeric nulls with 0 and text nulls with 'Unknown'
combined_df = combined_df.fillna({
    'numeric_column': 0,
    'text_column': 'Unknown'
})


In [ ]:
# Check total null values per column
null_counts = combined_df.isnull().sum()
print("Null values per column:")
print(null_counts[null_counts > 0])  # Only show columns with nulls

# Check total null values in entire dataset
total_nulls = combined_df.isnull().sum().sum()
print(f"\nTotal null values in dataset: {total_nulls}")

In [ ]:
# Show first 5 rows (default)
print(combined_df.head())

1. Setup & Data Preparation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your cleaned data
df = pd.read_excel('data/cleaned_combined_data.xlsx')

# Set visual style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))

2. Key Visualizations

A. Availability of Dementia Diagnostics by Country

In [ ]:
# Filter for diagnostic availability
diagnostics_df = df[df['IndicatorCode'] == 'GDO_q7x1_AVAIL'].dropna(subset=['Value'])

# Plot
plt.figure(figsize=(15, 8))
sns.countplot(data=diagnostics_df, y='Country', hue='Value', palette='viridis')
plt.title('Dementia Diagnostic Availability by Country')
plt.xlabel('Count')
plt.ylabel('Country')
plt.legend(title='Availability')
plt.tight_layout()
plt.show()

B. Distribution of Healthcare Professionals

In [ ]:

# 1. Filter and clean the data
professionals = df[df['IndicatorCode'].isin(['GDO_q6x1_2', 'GDO_q6x1_3', 'MH_6'])].copy()

# Ensure numeric values and drop NAs
professionals['FactValueNumeric'] = pd.to_numeric(professionals['FactValueNumeric'], errors='coerce')
professionals = professionals.dropna(subset=['FactValueNumeric'])

# 2. Create the visualization (without melting)
plt.figure(figsize=(16, 8))
sns.boxplot(
    data=professionals,
    x='Country',
    y='FactValueNumeric',
    hue='IndicatorCode',
    palette='coolwarm'
)

# 3. Format the plot
plt.xticks(rotation=90)
plt.title('Healthcare Professionals per 100k Population', pad=20)
plt.ylabel('Count per 100k')
plt.xlabel('Country')

# Improve legend
plt.legend(
    title='Professional Type',
    labels=['Neurologists (GDO_q6x1_2)', 'Geriatricians (GDO_q6x1_3)', 'Psychiatrists (MH_6)'],
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

plt.tight_layout()
plt.show()

C. Implementation of Dementia Guidelines (Heatmap)

In [ ]:
# Filter for guideline data
guidelines = df[df['IndicatorCode'].str.contains('GDO_q4x1')].pivot_table(
    index='Country',
    columns='Indicator',
    values='Value',
    aggfunc='first'
)

# Plot
plt.figure(figsize=(16, 10))
sns.heatmap(guidelines.notnull().astype(int), cmap='Blues', cbar=False)
plt.title('Implementation of Dementia Guidelines (1 = Exists)')
plt.show()

D. Time Trends (If Date Column Exists)

In [ ]:
if 'DateModified' in df:
    df['Year'] = df['DateModified'].dt.year
    yearly_counts = df.groupby(['Year', 'IndicatorCode']).size().unstack()

    yearly_counts.plot(kind='area', stacked=True, figsize=(14, 7))
    plt.title('Reported Indicators Over Time')
    plt.ylabel('Count')
    plt.show()

3. Advanced Insights

A. Correlation Matrix (Numeric Features)

In [ ]:
numeric_df = df.select_dtypes(include=['float64', 'int64'])
plt.figure(figsize=(12, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', center=0)
plt.title('Correlation Between Numeric Indicators')
plt.show()

B. Regional Comparison (Boxplot)

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='ParentLocation', y='FactValueNumeric', palette='Set3')
plt.xticks(rotation=45)
plt.title('Regional Distribution of Key Metrics')
plt.show()

4. Exporting Visualizations

In [ ]:
# Save all figures
for i, fig in enumerate(plt.get_fignums()):
    plt.figure(fig)
    plt.savefig(f'plot_{i}.png', dpi=300, bbox_inches='tight')

In [ ]:

# Load the data
data = pd.read_csv('data/cd4fdb4d-4725-46d4-bb53-5d4e238e3ce2.csv')

# Clean and prepare the data
# Extract relevant columns
df = data[['Location', 'Value']].copy()

# Convert "Yes" to 364 and "No" to 0 for visualization
df['Facility_Count'] = df['Value'].apply(lambda x: 364 if x == 'Yes' else (0 if x == 'No' else pd.NA))
df['Has_Facility'] = df['Value'].apply(lambda x: 'Available' if x == 'Yes' else 'Not Available')

# Sort by Facility_Count for better visualization
df = df.sort_values('Facility_Count', ascending=False)

# Plotting
plt.figure(figsize=(15, 8))

# Option 1: Bar plot with counts (though most are 364)
ax = sns.barplot(data=df, x='Location', y='Facility_Count', hue='Has_Facility',
                 dodge=False, palette={'Available': 'blue', 'Not Available': 'red'})

plt.title('Dementia Care Facilities (Hospitals) by Country (2017)', fontsize=16)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Number of Facilities (or Availability)', fontsize=12)
plt.xticks(rotation=90)
plt.tight_layout()

# Add annotations for the "No" case
for p in ax.patches:
    if p.get_height() == 0:
        ax.annotate('No', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.show()

# Print inference
print("\nInference:")
print("1. Most countries show 'Available' with a count of 364, suggesting this might be a standardized response.")
print("2. Türkiye is the only country showing 'Not Available', but with an unusual value (823).")
print("3. The uniformity of 364 across countries indicates this might represent facility availability rather than exact counts.")
print("4. All regions (Europe, Americas, Western Pacific, etc.) have representation in dementia care facilities.")
print("5. Data from 2017 may not reflect current dementia care infrastructure.")
print("6. No countries report complete absence of facilities, indicating global recognition of dementia care needs.")
print("7. The binary nature of responses limits deeper quantitative analysis.")
print("8. Further data on facility types, capacities, or patient numbers would enable more meaningful comparisons.")
print("9. The outlier (Türkiye) suggests possible data entry differences in reporting methodology.")
print("10. Despite limitations, the data shows dementia care is widely acknowledged as a health priority.")

In [ ]:
plt.figure(figsize=(15, 8))
sns.countplot(data=df, x='Location', hue='Has_Facility',
              palette={'Available': 'blue', 'Not Available': 'red'})
plt.title('Availability of Dementia Care Facilities by Country (2017)')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()